# 03_train_model_A — Train resnet50

Train a pretrained **resnet50** with transfer learning. Crash-resilient:

- Every epoch: training history JSON + epoch CSV → Drive
- Every best val_acc improvement: full training state checkpoint → Drive
- Resume support: if a checkpoint exists, set `RESUME = True` to continue

Output:
- `results/metrics/resnet50_model_card.{md,json}` (committed to GitHub)
- `pk_politicians_results/checkpoints/resnet50_best.pth` (Drive)
- `pk_politicians_results/logs/resnet50_history.json` (Drive)
- `pk_politicians_results/logs/resnet50_epoch_log.csv` (Drive)

## 1. Environment setup

This notebook is designed for **Google Colab (A100)**. The setup cell below:

1. Mounts Google Drive
2. Clones or updates the GitHub repo
3. Installs requirements
4. Adds the repo root to `sys.path` so `from src...` works

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/musarashid49/Image-Classification-with-CNN.git"
REPO_DIR = Path("/content/Image-Classification-with-CNN")

# 1. Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab — skipping drive mount.")

# 2. Clone or pull
if REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    print(f"Cloning {REPO_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# 3. Make repo importable
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Install requirements (Colab usually has torch already)
req = REPO_DIR / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

## 2. Config — edit here to switch model or hyperparameters

In [ ]:
# ---- Model & training config (single source of truth for this notebook) ----
MODEL_NAME = "resnet50"        # see src/models.py for options
EPOCHS = 35
LR_HEAD = 0.001
LR_BACKBONE = 0.0001
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
EARLY_STOPPING_PATIENCE = 8
SCHEDULER_PATIENCE = 3
DROPOUT_HEAD = 0.2
BATCH_SIZE = 32
RESUME = False           # set True to resume from existing checkpoint on Drive
FORCE_RETRAIN = False    # set True to ignore an existing _best checkpoint and retrain

## 3. Imports & setup

In [ ]:
import time
import torch
from dataclasses import asdict

from config.config import (
    CLASS_NAMES, DATASET_DIR,
    LOCAL_RESULTS, LOCAL_CHECKPOINTS, LOCAL_METRICS,
    RESULTS_DRIVE, CHECKPOINTS_DRIVE,
    SEED, IMG_SIZE,
)
from src.utils import (
    set_seed, get_device, gpu_info, ensure_dir,
    save_model_card, ExperimentLogger,
)
from src.dataset import build_dataloaders
from src.models import build_model, get_param_groups
from src.train import Trainer, TrainConfig

set_seed(SEED)
device = get_device()
print(f"Device: {device}")
print(f"GPU: {gpu_info() or 'CPU only'}")

## 4. Data loaders

In [ ]:
train_loader, val_loader, test_loader, class_to_idx = build_dataloaders(
    dataset_root=DATASET_DIR,
    batch_size=BATCH_SIZE,
)
# The order of CLASS_NAMES matches the order ImageFolder discovered (verified
# by build_dataloaders), so model output index i corresponds to CLASS_NAMES[i].
print(f"train batches: {len(train_loader)}, "
      f"val batches: {len(val_loader)}, "
      f"test batches: {len(test_loader)}")
print(f"class_to_idx: {class_to_idx}")

## 5. Build model

In [ ]:
model, info = build_model(MODEL_NAME, dropout=DROPOUT_HEAD)
param_groups = get_param_groups(
    model,
    lr_head=LR_HEAD, lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
)
print(f"Model: {info['model_name']}")
print(f"Total params: {info['total_params']:,} ({info['total_params_M']}M)")
print(f"Trainable: {info['trainable_params']:,}")

## 6. Resume / skip logic

In [ ]:
ckpt_path_drive = CHECKPOINTS_DRIVE / f"{MODEL_NAME}_best.pth"
resume_from = None

if ckpt_path_drive.exists():
    print(f"Existing checkpoint found at {ckpt_path_drive}")
    if FORCE_RETRAIN:
        print("FORCE_RETRAIN=True — will retrain from scratch.")
    elif RESUME:
        print("RESUME=True — continuing training from this checkpoint.")
        resume_from = ckpt_path_drive
    else:
        print("Neither RESUME nor FORCE_RETRAIN is True. The trainer will only")
        print("overwrite the checkpoint if a new epoch beats the current val_acc.")
        print("If the existing run is complete, set RESUME=False, run, and watch.")

## 7. Trainer config + run

In [ ]:
cfg = TrainConfig(
    model_name=MODEL_NAME,
    epochs=EPOCHS,
    lr_head=LR_HEAD,
    lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
    label_smoothing=LABEL_SMOOTHING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    scheduler_patience=SCHEDULER_PATIENCE,
)

trainer = Trainer(
    model=model,
    train_loader=train_loader, val_loader=val_loader,
    device=device, cfg=cfg,
    param_groups=param_groups,
    local_results_dir=LOCAL_RESULTS,
    drive_results_dir=RESULTS_DRIVE,
    resume_from=resume_from,
)
run = trainer.run()
print(f"\nTraining complete. best_val_acc={run['best_val_acc']:.4f} "
      f"in {run['duration_seconds']:.1f}s "
      f"({run['epochs_completed']} epochs)")

## 8. Quick test-set evaluation + model card

In [ ]:
from src.evaluate import (
    evaluate_and_save, predict_on_loader, compute_metrics
)
from config.config import LOCAL_PLOTS, REPORT_FIGURES, LOCAL_METRICS

# Load best weights (in case last epoch wasn't best)
from src.train import load_model_for_inference
model = load_model_for_inference(model, trainer.best_path_local, device)

metrics = evaluate_and_save(
    model=model,
    test_loader=test_loader,
    device=device,
    class_names=CLASS_NAMES,
    model_name=MODEL_NAME,
    plots_dir=LOCAL_PLOTS,
    report_dir=REPORT_FIGURES,
    metrics_dir=LOCAL_METRICS,
    history=trainer.history,
)
print(f"\nTest metrics for {MODEL_NAME}:")
print(f"  accuracy:        {metrics['accuracy']:.4f}")
print(f"  macro precision: {metrics['macro_precision']:.4f}")
print(f"  macro recall:    {metrics['macro_recall']:.4f}")
print(f"  macro F1:        {metrics['macro_f1']:.4f}")
print(f"  weighted F1:     {metrics['weighted_f1']:.4f}")

In [ ]:
# Save the model card (markdown + json) into the GitHub repo
card_paths = save_model_card(
    model_name=MODEL_NAME,
    model_info=info,
    training_config=asdict(cfg),
    final_metrics={
        "test_accuracy": metrics["accuracy"],
        "test_macro_precision": metrics["macro_precision"],
        "test_macro_recall": metrics["macro_recall"],
        "test_macro_f1": metrics["macro_f1"],
        "test_weighted_f1": metrics["weighted_f1"],
        "best_val_acc": run["best_val_acc"],
        "epochs_completed": run["epochs_completed"],
    },
    training_duration_seconds=run["duration_seconds"],
    checkpoint_drive_path=str(trainer.best_path_drive),
    output_dir=LOCAL_METRICS,
    notebook_name="03_train_model_A.ipynb",
    notes="Trained on dataset_resplit (75/15/10 split, per-class).",
)
print(f"Model card: {card_paths['md']}")
print(f"Model card: {card_paths['json']}")

# Append to experiment log
ExperimentLogger(LOCAL_METRICS / "experiment_log.jsonl").log({
    "model": MODEL_NAME,
    "test_accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "epochs": run["epochs_completed"],
    "duration_seconds": run["duration_seconds"],
})

## 9. (Manual) commit results to GitHub

After this notebook completes, commit the new files in your local repo:

```bash
cd Image-Classification-with-CNN
git add results/metrics/ results/plots/ report/figures/
git commit -m "feat: training run for {MODEL}"
git push origin main
```